In [ ]:
from moabb.datasets import BNCI2014_001
import moabb.paradigms as mp
from sklearn.pipeline import make_pipeline
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from moabb.evaluations import WithinSessionEvaluation
import random


In [ ]:
dataset = BNCI2014_001()
pipeline = make_pipeline(
            CSP(n_components=8),
            LDA()
        )

In [ ]:

channels=['C2', 'C3', 'C4', 'C5', 'C6', 'CP1', 'CP2', 'CP3', 
        'CP4', 'CPz', 'Cz', 'FC2','FC3', 'FC4', 'FCz', 'P1', 
        'P2', 'POz', 'Pz','C1', 'FC1','Fz']


In [ ]:

all_scaled = []
used_combinations = set()

base_channels = [
    'C2', 'C3', 'C4', 'C5', 'C6', 'CP1', 'CP2', 'CP3',
    'CP4', 'CPz', 'Cz', 'FC2', 'FC3', 'FC4', 'FCz',
    'P1', 'P2', 'POz', 'Pz', 'C1', 'FC1', 'Fz'
]

while len(all_scaled) < 10:
    num_inc=random.randint(1, 4)
    scaled = sorted(random.sample(base_channels, num_inc))
    key = tuple(scaled)

    if key in used_combinations:
        continue

    used_combinations.add(key)
    all_scaled.append(scaled)

print(all_scaled)

In [ ]:
def scale_EEG(names):
    

In [ ]:
all_results = []
all_avgs = []

for scaled in all_scaled:

    channels = [ch for ch in base_channels if ch not in scaled]
    

    paradigm = mp.MotorImagery(
        scorer=["accuracy", "balanced_accuracy", "f1_macro"],
        channels=channels
    )

    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=[dataset],
        overwrite=True,
        hdf5_path=None
    )

    results = evaluation.process({"csp+lda": pipeline})
    all_results.append(results)

    
    avgRes=[]
    avgRes=pd.DataFrame(avgRes)
    avgRes['Accuracy']=results.groupby('session')['score_accuracy'].mean()
    avgRes['Balanced Accuracy']=results.groupby('session')['score_balanced_accuracy'].mean()
    avgRes['score_f1_macro']=results.groupby('session')['score_f1_macro'].mean()


    all_avgs.append(avgRes)